# 360-Vid2Splat · Ücretsiz GPU ile videodan gezilebilir 3D sahne

Bu not defteri bir **360° videoyu** (equirectangular, örn. Insta360'tan dışa aktarılmış) veya **normal telefon videosunu** alır ve web'de WASD ile gezilebilen bir 3D Gaussian Splat sahnesine çevirir.

**Başlamadan önce:** `Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU` seçin.

| Adım | Ne yapar | T4'te yaklaşık süre |
|---|---|---|
| 1. Kurulum | COLMAP (GPU), gsplat, Node.js | 8–12 dk (gsplat ilk seferde derlenir) |
| 2. SfM | En net kareleri seçer, kamera konumlarını bulur | 5–20 dk |
| 3. Eğitim | 3D Gaussian Splatting | 15–30 dk (15.000 adım) |
| 4. Dışa aktarma | Temizlik + SPZ sıkıştırma | 1 dk |

**İyi çekim için:** yavaş yürü, kamerayı başının üstünde sabit tut, aynı yerden iki kez geçmekten çekinme, hareketli insan ve aynalardan kaçın. 1–2 dakikalık video yeterli.

In [ ]:
#@title 1. GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Kurulum (8–12 dk)
REPO_URL = "https://github.com/Oguzhan61-dilmac/360-vid2splat.git"  #@param {type:"string"}
import os
if not os.path.exists("/content/360-vid2splat"):
    !git clone -q {REPO_URL} /content/360-vid2splat
%cd /content/360-vid2splat
!pip install -q numpy opencv-python-headless scipy tqdm ninja
!pip install -q pycolmap-cuda12==4.2.1
!pip install -q --no-build-isolation gsplat==1.5.3
!node -v || (apt-get -qq update && apt-get -qq install -y nodejs npm)

# gsplat CUDA çekirdeklerini şimdi derle (ilk seferde birkaç dakika sürer)
import torch
from gsplat import rasterization
n = 16
img, _, _ = rasterization(
    torch.randn(n, 3, device="cuda"), torch.randn(n, 4, device="cuda"), torch.rand(n, 3, device="cuda") * 0.1,
    torch.rand(n, device="cuda"), torch.rand(n, 3, device="cuda"),
    torch.tensor([[[1., 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 5], [0, 0, 0, 1]]], device="cuda"),
    torch.tensor([[[100., 0, 50], [0, 100, 50], [0, 0, 1]]], device="cuda"), 100, 100)
import pycolmap
print("Hazır. COLMAP GPU:", pycolmap.has_cuda, "| gsplat çıktı:", tuple(img.shape))

## Videoyu yükle

İki yol var: aşağıdaki hücreyle bilgisayarından yükle ya da büyük dosyalar için Google Drive'ı bağlayıp `VIDEO` yoluna Drive'daki dosyayı yaz (örn. `/content/drive/MyDrive/cekimler/oda.mp4`).

In [ ]:
#@title 3. Video yükle
from google.colab import files
uploaded = files.upload()
VIDEO = "/content/" + next(iter(uploaded))
print("Video:", VIDEO)

In [ ]:
#@title (İsteğe bağlı) Google Drive'dan video kullan
from google.colab import drive
drive.mount("/content/drive")
VIDEO = "/content/drive/MyDrive/oda.mp4"  #@param {type:"string"}

### 360 kameran yoksa: hazır 360° veriyle dene

Aşağıdaki hücre, 8K 360° drone videosundan çıkarılmış **Big Sur** panoramalarını (Jamesbass/bigsur-360-colmap, CC BY 4.0) indirir. Klasör doğrudan girdi olarak kullanılır; drone yüksekte uçtuğu için 4. adımda `CEKIM_YUKSEKLIGI` değerini **20** yap.

In [ ]:
#@title (İsteğe bağlı) Örnek 360° veri indir: Big Sur
#@markdown Toplam 773 panorama var; her kaçıncısını alalım? (4 → ~190 panorama, ~850 MB)
HER_KACINCI = 4  #@param {type:"integer"}
!pip install -q huggingface_hub
from huggingface_hub import HfApi, hf_hub_download
repo = "Jamesbass/bigsur-360-colmap"
names = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.startswith("images/"))
secilen = names[::HER_KACINCI]
for i, f in enumerate(secilen):
    hf_hub_download(repo, f, repo_type="dataset", local_dir="/content/bigsur")
    if i % 20 == 0:
        print(f"{i + 1}/{len(secilen)} panorama indirildi")
VIDEO = "/content/bigsur/images"
print("Girdi klasörü:", VIDEO)


In [ ]:
#@title 4. Kamera konumları (SfM)
SAHNE_ADI = "oda"  #@param {type:"string"}
MOD = "auto"  #@param ["auto", "360", "perspective"]
#@markdown Kameranın yerden yüksekliği (metre). 360 kamera başın üstünde ~1.9, göğüs hizasında ~1.4; telefon ~1.5.
CEKIM_YUKSEKLIGI = 1.7  #@param {type:"number"}
#@markdown Kaç saniyede bir kare seçilsin ve en fazla kaç kare kullanılsın (az kare = hızlı ama eksik sahne).
ARALIK = 0.5  #@param {type:"number"}
MAKS_KARE = 160  #@param {type:"integer"}
OUT = f"/content/output/{SAHNE_ADI}"
!python -m vid2splat sfm "{VIDEO}" --out "{OUT}" --mode {MOD} --capture-height {CEKIM_YUKSEKLIGI} --interval {ARALIK} --max-frames {MAKS_KARE}

In [ ]:
#@title 5. Eğitim + web'e dışa aktarma
#@markdown 15.000 adım T4'te ~20 dk; 30.000 en iyi kalite ama iki kat sürer.
ADIM = 15000  #@param {type:"integer"}
MAKS_SPLAT = 1000000  #@param {type:"integer"}
!python -m vid2splat train "{OUT}" --steps {ADIM} --max-splats {MAKS_SPLAT}

In [ ]:
#@title 6. Sonuçlara bak (soldaki gerçek kare, sağdaki model)
import json, glob
from IPython.display import Image, display
print(json.dumps(json.load(open(f"{OUT}/splat/metrics.json")), indent=2))
for p in sorted(glob.glob(f"{OUT}/splat/previews/*.jpg"))[:3]:
    display(Image(p, width=900))

## Siteye ekle

İndirdiğin zip'i repo içinde `web/scenes/<sahne-adı>/` klasörüne aç (içinde `scene.spz` ve `scene.json` olacak) ve `web/scenes/index.json` listesine bir satır ekle:

```json
{"id": "oda", "name": "Bizim oda", "path": "scenes/oda/"}
```

`main` dalına push ettiğinde Vercel siteyi otomatik olarak yeniden yayınlar (https://360-vid2splat.vercel.app). Dosya 100 MB'ı geçerse GitHub kabul etmez; o zaman `--max-splats 500000` ile yeniden eğit ya da dosyayı Hugging Face'e koyup görüntüleyiciyi `?scene=<klasör-adresi>` ile aç.

In [ ]:
#@title 7. Web dosyalarını indir
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{SAHNE_ADI}_web", "zip", f"{OUT}/web")
files.download(zip_path)